# HDMI Video Pipeline

## Imports

In [1]:
from pynq import Overlay
from pynq.lib.video import *

from model.model import MiniFCOSFaceV1
from accelerator.FCOSAccelerator import FCOSAccelerator

from model.postprocessing.decode_raw_output import decode_predictions
from model.preprocessing.preprocess_frame import preprocess_frame 

import cv2
import numpy as np

import time
from collections import deque

## Overlay, IP

In [2]:
overlay = Overlay("./bitstream/fcos_accelerate.bit")
ip = overlay.face_accelerate_0

## Instantiating HDMI in and out 

In [3]:
hdmi_in = overlay.video.hdmi_in
hdmi_out = overlay.video.hdmi_out
    
hdmi_in.configure(PIXEL_RGB)
hdmi_out.configure(hdmi_in.mode, PIXEL_RGB)

hdmi_in.start()
hdmi_out.start()

## Model - instance

In [4]:
weights_path = "./model/weights/minifcos_face_v1_weights.npz"

weights_npz = "./model/weights/minifcos_face_v1_folded_weights.npz"

model_numpy = MiniFCOSFaceV1(
    weights_path
)

model_fpga = FCOSAccelerator(ip, weights_npz, verbose = False)

## Video Test

### Numpy

In [ ]:
DURATION_S = 30.0
sw_times = []
n_runs = 0

start_wall = time.perf_counter()

while (time.perf_counter() - start_wall) < DURATION_S:
    frame = hdmi_in.readframe()

    processed_frame = preprocess_frame(frame)

    t0 = time.perf_counter()
    model_result = model_numpy(processed_frame)
    t1 = time.perf_counter()
    sw_times.append(t1 - t0)
    n_runs += 1

    boxes_numpy, scores_numpy = decode_predictions(model_result)

    threshold = 0.20
    MODEL_W = 320
    MODEL_H = 320
    FRAME_H, FRAME_W = frame.shape[:2]

    scale = min(MODEL_W / FRAME_W, MODEL_H / FRAME_H)
    resized_w = int(round(FRAME_W * scale))
    resized_h = int(round(FRAME_H * scale))
    pad_left = (MODEL_W - resized_w) // 2
    pad_top = (MODEL_H - resized_h) // 2

    for box, score in zip(boxes_numpy, scores_numpy):
        if score < threshold:
            continue
        x1, y1, x2, y2 = box
        x1 = int(np.clip((x1 - pad_left) / scale, 0, FRAME_W - 1))
        y1 = int(np.clip((y1 - pad_top) / scale, 0, FRAME_H - 1))
        x2 = int(np.clip((x2 - pad_left) / scale, 0, FRAME_W - 1))
        y2 = int(np.clip((y2 - pad_top) / scale, 0, FRAME_H - 1))
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 3)
        cv2.putText(frame, f"Face: {score:.2f}", (x1, max(y1 - 10, 25)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA)

    hdmi_out.writeframe(frame)

sw_times = np.array(sw_times) * 1000  # ms

print(f"\n=== SW (numpy) metrics ===")
print(f"duration    = {DURATION_S:.0f} s")
print(f"n runs      = {n_runs}")
print(f"mean        = {sw_times.mean():.3f} ms")
print(f"std         = {sw_times.std():.3f} ms")
print(f"min         = {sw_times.min():.3f} ms")
print(f"max         = {sw_times.max():.3f} ms")
print(f"FPS (1/mean)= {1000.0 / sw_times.mean():.2f}")

### Accelerator

In [8]:
import time
import numpy as np
import cv2

DURATION_S = 30.0
hw_times = []
n_runs = 0

start_wall = time.perf_counter()

while (time.perf_counter() - start_wall) < DURATION_S:
    frame = hdmi_in.readframe()

    processed_frame = preprocess_frame(frame)

    t0 = time.perf_counter()
    model_result = model_fpga.run(processed_frame)
    t1 = time.perf_counter()
    hw_times.append(t1 - t0)
    n_runs += 1

    boxes_numpy, scores_numpy = decode_predictions(model_result)

    threshold = 0.20
    MODEL_W = 320
    MODEL_H = 320
    FRAME_H, FRAME_W = frame.shape[:2]

    scale = min(MODEL_W / FRAME_W, MODEL_H / FRAME_H)
    resized_w = int(round(FRAME_W * scale))
    resized_h = int(round(FRAME_H * scale))
    pad_left = (MODEL_W - resized_w) // 2
    pad_top = (MODEL_H - resized_h) // 2

    for box, score in zip(boxes_numpy, scores_numpy):
        if score < threshold:
            continue
        x1, y1, x2, y2 = box
        x1 = int(np.clip((x1 - pad_left) / scale, 0, FRAME_W - 1))
        y1 = int(np.clip((y1 - pad_top) / scale, 0, FRAME_H - 1))
        x2 = int(np.clip((x2 - pad_left) / scale, 0, FRAME_W - 1))
        y2 = int(np.clip((y2 - pad_top) / scale, 0, FRAME_H - 1))
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 3)
        cv2.putText(frame, f"Face: {score:.2f}", (x1, max(y1 - 10, 25)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA)

    hdmi_out.writeframe(frame)

hw_times = np.array(hw_times) * 1000  # ms

print(f"\n=== HW (FPGA) metrics ===")
print(f"duration    = {DURATION_S:.0f} s")
print(f"n runs      = {n_runs}")
print(f"mean        = {hw_times.mean():.3f} ms")
print(f"std         = {hw_times.std():.3f} ms")
print(f"min         = {hw_times.min():.3f} ms")
print(f"max         = {hw_times.max():.3f} ms")
print(f"FPS (1/mean)= {1000.0 / hw_times.mean():.2f}")


=== HW (FPGA) metrics ===
duration    = 30 s
n runs      = 38
mean        = 704.428 ms
std         = 6.373 ms
min         = 700.928 ms
max         = 735.417 ms
FPS (1/mean)= 1.42


### Comparison

In [7]:
if 'sw_times' in dir() and 'hw_times' in dir():
    speedup = sw_times.mean() / hw_times.mean()
    print(f"SW mean = {sw_times.mean():.3f} ms   ({1000/sw_times.mean():.2f} FPS)")
    print(f"HW mean = {hw_times.mean():.3f} ms   ({1000/hw_times.mean():.2f} FPS)")
    print(f"Speedup (SW/HW) = {speedup:.3f}x  ({'HW BRZI' if speedup > 1 else 'HW SPORIJI'})")

=== Usporedba ===
SW mean = 853.895 ms   (1.17 FPS)
HW mean = 704.162 ms   (1.42 FPS)
Speedup (SW/HW) = 1.213x  (HW BRZI)


## Pipeline - Inference

HDMI IN -> READ FRAME -> PREPROCESS FRAME -> PASS FRAME THROUGH MODEL -> POSTPROCESS RESULT -> DRAW RECTANGLES -> HDMI OUT

In [9]:
#LOOP
while(True):    
    frame = hdmi_in.readframe()
    
    #preprocess call
    processed_frame = preprocess_frame(frame)
    
    #model call (preprocess)
    #model_result = model_numpy(processed_frame)
    model_result = model_fpga.run(processed_frame)
    
    #postprocess (model_result)
    boxes_numpy, scores_numpy = decode_predictions(model_result)
    
    threshold = 0.20
    MODEL_W = 320
    MODEL_H = 320
    FRAME_H, FRAME_W = frame.shape[:2]

    scale = min(MODEL_W / FRAME_W, MODEL_H / FRAME_H)
    resized_w = int(round(FRAME_W * scale))
    resized_h = int(round(FRAME_H * scale))
    pad_left = (MODEL_W - resized_w) // 2
    pad_top = (MODEL_H - resized_h) // 2

    for box, score in zip(boxes_numpy, scores_numpy):
        if score < threshold:
            continue
        x1, y1, x2, y2 = box
        x1 = int(np.clip((x1 - pad_left) / scale, 0, FRAME_W - 1))
        y1 = int(np.clip((y1 - pad_top) / scale, 0, FRAME_H - 1))
        x2 = int(np.clip((x2 - pad_left) / scale, 0, FRAME_W - 1))
        y2 = int(np.clip((y2 - pad_top) / scale, 0, FRAME_H - 1))
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 3)
        cv2.putText(frame, f"Face: {score:.2f}", (x1, max(y1 - 10, 25)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA)

    #sljedeći 
    hdmi_out.writeframe(frame)



KeyboardInterrupt



In [10]:
hdmi_in.close()
hdmi_out.close()
model_fpga.close()